# HiProbCBM — Setup & Training di Colab (dataset CUB-200-2011)

Notebook ini menyiapkan struktur dataset yang dibutuhkan `hiprobcbm/data/cub.py`, lalu menjalankan training.

**Struktur akhir yang dibutuhkan kode:**
```
datasets/CUB_200_2011/
  images/<kelas>/<file>.jpg      <- dari Drive kamu (raw CUB_200_2011)
  metadata/train.pkl             <- diambil dari repo HiCEM kalau belum ada di Drive
  metadata/val.pkl
  metadata/test.pkl
```

**Sebelum jalan:** edit `DRIVE_CUB_ROOT` di sel 2 supaya menunjuk ke folder `CUB_200_2011` di Drive kamu (yang isinya `images/`, `attributes/`, `bounding_boxes.txt`, dst — hasil download resmi Caltech).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set path ke folder CUB_200_2011 di Drive kamu

**Wajib diedit** — sesuaikan dengan lokasi folder di Drive kamu (klik kanan folder > Copy path, atau cek lewat sidebar Files).

In [ ]:
import os

# TODO: sesuaikan path ini dengan lokasi folder CUB_200_2011 di Drive kamu
DRIVE_CUB_ROOT = "/content/drive/MyDrive/CUB_200_2011"

assert os.path.isdir(DRIVE_CUB_ROOT), f"Folder tidak ditemukan: {DRIVE_CUB_ROOT} -- cek lagi path-nya di sidebar Files Colab"
assert os.path.isdir(os.path.join(DRIVE_CUB_ROOT, "images")), "Folder 'images' tidak ada di dalam DRIVE_CUB_ROOT -- pastikan ini folder CUB_200_2011 yang benar"
print("OK, DRIVE_CUB_ROOT valid:", DRIVE_CUB_ROOT)

## 3. Clone repo HiProbCBM & install

In [ ]:
%cd /content
!test -d HiProbCBM || git clone https://github.com/Azhimanich/HiProbCBM.git
%cd /content/HiProbCBM
!pip install -e . -q

## 4. Siapkan metadata (train/val/test.pkl)

Kode `hiprobcbm/data/cub.py` butuh `metadata/train.pkl`, `val.pkl`, `test.pkl` (hasil preprocessing 112-konsep ala Koh dkk. 2020) di dalam `DRIVE_CUB_ROOT`.

- Kalau file itu **sudah** kamu upload ke Drive -> sel ini cuma skip, langsung dipakai.
- Kalau **belum** -> sel ini ambil otomatis dari repo referensi `HiCEM` (`OscarPi/cem-concept-discovery`, folder `splits/CUB/class_attr_data_10/`) dan menyimpannya ke Drive kamu supaya tidak perlu diulang tiap sesi Colab.

In [ ]:
import pathlib, shutil

METADATA_DIR = pathlib.Path(DRIVE_CUB_ROOT) / "metadata"
NEEDED = ["train.pkl", "val.pkl", "test.pkl"]

missing = [f for f in NEEDED if not (METADATA_DIR / f).exists()]

if not missing:
    print("Metadata sudah lengkap di Drive, tidak perlu ambil ulang:", METADATA_DIR)
else:
    print("Belum lengkap, ambil dari HiCEM (GitHub). File yang kurang:", missing)
    %cd /content
    !test -d HiCEM_ref || git clone --depth 1 https://github.com/OscarPi/cem-concept-discovery.git HiCEM_ref
    METADATA_DIR.mkdir(parents=True, exist_ok=True)
    for f in missing:
        src = pathlib.Path("/content/HiCEM_ref/splits/CUB/class_attr_data_10") / f
        shutil.copy(src, METADATA_DIR / f)
        print("disalin:", src, "->", METADATA_DIR / f)
    %cd /content/HiProbCBM

## 5. Normalisasi `img_path` di dalam pkl

File pkl dari HiCEM/ProbCBM menyimpan `img_path` dengan prefix path lama dari server penulis paper aslinya (mis. `/juice/scr/.../CUB_200_2011/images/...`), bukan path relatif `<kelas>/<file>.jpg` yang diasumsikan `hiprobcbm/data/cub.py`. Sel ini membersihkan itu sekali saja, ditulis ulang ke Drive supaya tidak perlu diulang tiap sesi.

In [ ]:
import pickle

MARKER = "CUB_200_2011/images/"

def normalize_img_path(p: str) -> str:
    return p.split(MARKER, 1)[1] if MARKER in p else p

for fname in NEEDED:
    fpath = METADATA_DIR / fname
    with open(fpath, "rb") as f:
        records = pickle.load(f)

    n_changed = sum(1 for r in records if MARKER in r["img_path"])
    if n_changed == 0:
        print(f"{fname}: sudah relatif, tidak ada yang diubah ({len(records)} record)")
        continue

    for r in records:
        r["img_path"] = normalize_img_path(r["img_path"])

    with open(fpath, "wb") as f:
        pickle.dump(records, f)
    print(f"{fname}: {n_changed}/{len(records)} img_path dinormalisasi & disimpan ulang ke Drive")

## 6. Rakit struktur `datasets/CUB_200_2011/` (symlink ke Drive, tanpa copy gambar)

In [ ]:
import os

WORK_ROOT = pathlib.Path("/content/HiProbCBM/datasets/CUB_200_2011")
WORK_ROOT.parent.mkdir(parents=True, exist_ok=True)

if WORK_ROOT.is_symlink() or WORK_ROOT.exists():
    if WORK_ROOT.is_symlink():
        WORK_ROOT.unlink()
    else:
        raise RuntimeError(f"{WORK_ROOT} sudah ada dan bukan symlink -- cek manual dulu")

os.symlink(DRIVE_CUB_ROOT, WORK_ROOT, target_is_directory=True)
print("Symlink dibuat:", WORK_ROOT, "->", DRIVE_CUB_ROOT)

print("\nVerifikasi struktur:")
for sub in ["images", "metadata/train.pkl", "metadata/val.pkl", "metadata/test.pkl"]:
    p = WORK_ROOT / sub
    print(f"  [{'OK' if p.exists() else 'HILANG'}] {sub}")

## 7. Sanity check — buka satu gambar contoh lewat pkl

In [ ]:
import random
from PIL import Image

with open(WORK_ROOT / "metadata" / "train.pkl", "rb") as f:
    train_records = pickle.load(f)

print("Jumlah data train:", len(train_records))
sample = random.choice(train_records)
print("img_path :", sample["img_path"])
print("kelas    :", sample["class_label"])
print("n konsep :", len(sample["attribute_label"]))

img_path = WORK_ROOT / "images" / sample["img_path"]
img = Image.open(img_path).convert("RGB")
print("Gambar berhasil dibuka, ukuran:", img.size)
img

## 8. Dry-run 2 batch data ASLI lewat model asli

**Kenapa perlu:** README `HiProbCBM` sendiri bilang repo ini baru diverifikasi lewat 34 *smoke test* dengan data sintetis di CPU — **belum pernah** dijalankan pada CUB-200-2011 asli. Sel #5 tadi sudah menemukan & memperbaiki 1 bug nyata (`img_path`). Supaya tidak baru ketahuan ada bug setelah training 50 epoch jalan berjam-jam, sel ini menjalankan 2 batch data asli lewat `HiProbCBMStage1` sungguhan (dipanggil langsung, bukan lewat CLI) dan mengecek loss/shape-nya masuk akal, sebelum commit ke training penuh.

In [ ]:
import torch
from hiprobcbm.config import load_config, resolve_device
from hiprobcbm.data import build_dataset
from hiprobcbm.losses import hiprobcbm_stage1_loss
from hiprobcbm.models.hiprobcbm import HiProbCBMStage1

cfg = load_config("configs/cub_stage1.yaml")
device = resolve_device(0)
print("Device:", device)

dataset = build_dataset(cfg.dataset, **cfg.data.to_dict())
train_loader = dataset.get_dataloader("train", cfg.model.image_size, cfg.model.backbone, batch_size=4)

model = HiProbCBMStage1(
    backbone_name=cfg.model.backbone,
    num_concepts=dataset.num_concepts,
    concept_dim=cfg.model.concept_dim,
    pretrained=cfg.model.get("pretrained", True),
).to(device)

batch = next(iter(train_loader))
x = batch["image"].to(device)
c = batch["concepts"].to(device)
print("image batch shape :", x.shape)
print("concept batch shape:", c.shape, "(harus [batch, 112])")

out = model(x)
loss, components = hiprobcbm_stage1_loss(out.concept_probs, c, out.kl_loss, lambda_kl=cfg.train.get("lambda_kl", 5e-5))
loss.backward()

print("\nForward + backward SUKSES pada data asli.")
print("loss total:", loss.item(), "| komponen:", {k: v.item() for k, v in components.items()})
assert torch.isfinite(loss), "loss NaN/Inf -- cek learning rate / normalisasi data sebelum training penuh"

## 9. (Opsional) Jalankan smoke test dulu sebelum training penuh

In [ ]:
%cd /content/HiProbCBM
!pip install -e ".[dev]" -q
!python -m pytest tests/ -q

## 10. Jalankan training — Tahap 1

Default pakai backbone Inception-v3 (`configs/cub_stage1.yaml`). Ganti ke `configs/cub_stage1_resnet18.yaml` untuk backbone ResNet18 (konvensi ProbCBM).

In [ ]:
%cd /content/HiProbCBM
!python scripts/run_stage1.py --config configs/cub_stage1.yaml

## 11. Jalankan training — Tahap 2

Perlu `--stage1-log-dir` menunjuk ke output Tahap 1 di atas (cek path yang dicetak `run_stage1.py`, biasanya di `train_log/cub/...`).

In [ ]:
%cd /content/HiProbCBM
!python scripts/run_stage2.py --config configs/cub_stage2.yaml \
    --stage1-log-dir train_log/cub/cub_stage1_inception

## 12. Sambungkan `train_log/` ke Drive (persisten lintas sesi)

**Kenapa sebelum training, bukan sesudah:** `/content/HiProbCBM/train_log` ada di disk sementara Colab — hilang total begitu runtime disconnect (idle atau limit ~12 jam), termasuk di TENGAH salah satu dari 16 skenario di sel berikutnya. Kalau baru dicadangkan di akhir, checkpoint yang belum sempat disalin ikut hilang, dan sesi berikutnya tidak akan tahu ada progres sebelumnya — sel #13 akan mengulang semua dari nol. Sel ini menyambungkan `train_log/` sebagai symlink ke Drive (pola sama seperti `datasets/CUB_200_2011/` di sel #6) supaya **setiap checkpoint langsung tersimpan ke Drive saat itu juga**, dan tetap terdeteksi oleh logika resumable di sel #13 walau sesi Colab-nya beda.

In [ ]:
TRAIN_LOG_DRIVE = pathlib.Path(DRIVE_CUB_ROOT).parent / "HiProbCBM_results" / "train_log"
TRAIN_LOG_DRIVE.mkdir(parents=True, exist_ok=True)

TRAIN_LOG_LOCAL = pathlib.Path("/content/HiProbCBM/train_log")
if TRAIN_LOG_LOCAL.is_symlink():
    TRAIN_LOG_LOCAL.unlink()
elif TRAIN_LOG_LOCAL.exists():
    raise RuntimeError(f"{TRAIN_LOG_LOCAL} sudah ada dan bukan symlink -- cek manual dulu")

os.symlink(TRAIN_LOG_DRIVE, TRAIN_LOG_LOCAL, target_is_directory=True)
print("train_log/ tersambung ke Drive:", TRAIN_LOG_LOCAL, "->", TRAIN_LOG_DRIVE)
print("Semua checkpoint & pseudo_hierarchy.pt dari sini akan langsung masuk Drive, persisten lintas sesi.")

## 13. Jalankan SEMUA skenario (Tabel 4.7 + Tabel 4.8 + tambahan)

Sel #10 & #11 di atas cuma contoh 1 skenario cepat. Sel di bawah ini menjalankan **seluruh** skenario secara berurutan (urutan sudah memperhitungkan dependensi — Tahap 2 / ablasi baru jalan setelah Tahap 1 terkait selesai). Pastikan sel #12 sudah dijalankan lebih dulu supaya semua checkpoint langsung persisten ke Drive.

**Wajib — sesuai Tabel 4.7 & 4.8 proposal (Bab IV.7.3):**
1. CEM @ CUB @ Inception-v3
2. ProbCBM @ CUB @ Inception-v3
3. HiCEM @ PseudoKitchens @ CLIP ViT-L/14
4. HiProbCBM Tahap 1+2 @ CUB @ Inception-v3
5. HiProbCBM Tahap 1+2 @ PseudoKitchens @ CLIP ViT-L/14
6. HiProbCBM-A1 (tanpa learned attention) @ CUB
7. HiProbCBM-A2 (tanpa regularisasi KL) @ CUB

**Tambahan — dari diskusi sebelumnya, di luar tabel resmi (Keputusan Desain Eksperimen #1, #2, #4):**
8. CBM @ CUB @ Inception-v3 (baseline dasar, Subbab III.3.2)
9. ProbCBM anchor sanity-check (classifier ANCHOR asli, ResNet18 299×299 — verifikasi replikasi vs Tabel 1 paper asli Kim dkk. 2023)
10. ProbCBM @ CUB @ ResNet18 (dual-backbone, classifier terstandardisasi)
11. HiProbCBM Tahap 1+2 @ CUB @ ResNet18 (dual-backbone)
12. HiProbCBM-A3 Tahap 1+2 (varian SAE BatchTopK) @ CUB

Total **16 run**. Ini eksperimen berat — training penuh semuanya bisa makan **belasan jam sampai beberapa hari** GPU, jauh melebihi batas sesi gratis Colab (~12 jam). Makanya sel ini:
- **Resumable lintas sesi**: skip run yang log/checkpoint-nya sudah ada (cek `stageX_last.pth` / `pseudo_hierarchy.pt` di `train_log/`, yang sejak sel #12 sudah tersambung ke Drive), jadi aman dijalankan ulang tiap buka sesi Colab baru.
- **Tidak berhenti kalau 1 run gagal** — lanjut ke run berikutnya, error dicatat di ringkasan akhir supaya kamu tahu mana yang perlu diulang.
- Bisa dipersempit lewat `RUN_TAGS` di sel berikutnya (mis. cuma `{"wajib"}` dulu kalau waktu terbatas, kejar Tabel 4.7 & 4.8 dulu sebelum tambahan).

In [ ]:
import pathlib

# Persempit di sini kalau waktu/GPU terbatas:
#   {"wajib"}              -> cuma Tabel 4.7 + 4.8 (7 run)
#   {"wajib", "tambahan"}  -> semua 16 run (default)
RUN_TAGS = {"wajib", "tambahan"}

TL = "train_log"  # sesuai default cfg.get("log_dir", "train_log") di hiprobcbm/config.py

# Setiap entri: tag, deskripsi, command CLI, dan file "penanda selesai"
# (dicek sebelum run -> kalau sudah ada, run di-skip / resumable).
# Nama file checkpoint baseline = f"{cfg.baseline}_last.pth" (lihat
# hiprobcbm/engine/train_baseline.py -- variabel `name` diisi dari cfg.baseline,
# BUKAN string "baseline"), jadi menyesuaikan per baseline: cem/probcbm/hicem/cbm.
EXPERIMENTS = [
    # ---------------- Tabel 4.7: skenario pengujian utama (WAJIB) ----------------
    dict(tag="wajib", desc="CEM @ CUB @ Inception-v3",
         cmd="python scripts/run_baseline.py --config configs/baselines/cem_cub.yaml",
         done_marker=f"{TL}/cub/cem_cub_inception/cem_last.pth"),
    dict(tag="wajib", desc="ProbCBM @ CUB @ Inception-v3",
         cmd="python scripts/run_baseline.py --config configs/baselines/probcbm_cub.yaml",
         done_marker=f"{TL}/cub/probcbm_cub_inception/probcbm_last.pth"),
    dict(tag="wajib", desc="HiCEM @ PseudoKitchens @ CLIP ViT-L/14",
         cmd="python scripts/run_baseline.py --config configs/baselines/hicem_kitchens.yaml",
         done_marker=f"{TL}/kitchens/hicem_kitchens_clip/hicem_last.pth"),
    dict(tag="wajib", desc="HiProbCBM Tahap 1 @ CUB @ Inception-v3",
         cmd="python scripts/run_stage1.py --config configs/cub_stage1.yaml",
         done_marker=f"{TL}/cub/cub_stage1_inception/pseudo_hierarchy.pt"),
    dict(tag="wajib", desc="HiProbCBM Tahap 2 @ CUB @ Inception-v3",
         cmd=f"python scripts/run_stage2.py --config configs/cub_stage2.yaml "
             f"--stage1-log-dir {TL}/cub/cub_stage1_inception",
         done_marker=f"{TL}/cub/cub_stage2_inception/stage2_last.pth"),
    dict(tag="wajib", desc="HiProbCBM Tahap 1 @ PseudoKitchens @ CLIP ViT-L/14",
         cmd="python scripts/run_stage1.py --config configs/kitchens_stage1.yaml",
         done_marker=f"{TL}/kitchens/kitchens_stage1_clip/pseudo_hierarchy.pt"),
    dict(tag="wajib", desc="HiProbCBM Tahap 2 @ PseudoKitchens @ CLIP ViT-L/14",
         cmd=f"python scripts/run_stage2.py --config configs/kitchens_stage2.yaml "
             f"--stage1-log-dir {TL}/kitchens/kitchens_stage1_clip",
         done_marker=f"{TL}/kitchens/kitchens_stage2_clip/stage2_last.pth"),

    # ---------------- Tabel 4.8: studi ablasi (WAJIB) ----------------
    dict(tag="wajib", desc="HiProbCBM-A1 (tanpa learned attention) @ CUB",
         cmd=f"python scripts/run_ablation.py --variant a1 --config configs/cub_stage2.yaml "
             f"--stage1-log-dir {TL}/cub/cub_stage1_inception",
         done_marker=f"{TL}/cub/cub_stage2_inception_ablation_a1/stage2_last.pth"),
    dict(tag="wajib", desc="HiProbCBM-A2 (tanpa regularisasi KL) @ CUB",
         cmd=f"python scripts/run_ablation.py --variant a2 --config configs/cub_stage2.yaml "
             f"--stage1-log-dir {TL}/cub/cub_stage1_inception",
         done_marker=f"{TL}/cub/cub_stage2_inception_ablation_a2/stage2_last.pth"),

    # ---------------- Tambahan: di luar tabel resmi (hasil diskusi) ----------------
    dict(tag="tambahan", desc="CBM @ CUB @ Inception-v3 (baseline dasar)",
         cmd="python scripts/run_baseline.py --config configs/baselines/cbm_cub.yaml",
         done_marker=f"{TL}/cub/cbm_cub_inception/cbm_last.pth"),
    dict(tag="tambahan", desc="ProbCBM anchor sanity-check (ResNet18, classifier ANCHOR asli)",
         cmd="python scripts/run_baseline.py --config configs/baselines/probcbm_cub_anchor_sanity_check.yaml",
         done_marker=f"{TL}/cub/probcbm_cub_anchor_sanity_check/probcbm_last.pth"),
    dict(tag="tambahan", desc="ProbCBM @ CUB @ ResNet18 (dual-backbone)",
         cmd="python scripts/run_baseline.py --config configs/baselines/probcbm_cub_resnet18.yaml",
         done_marker=f"{TL}/cub/probcbm_cub_resnet18/probcbm_last.pth"),
    dict(tag="tambahan", desc="HiProbCBM Tahap 1 @ CUB @ ResNet18 (dual-backbone)",
         cmd="python scripts/run_stage1.py --config configs/cub_stage1_resnet18.yaml",
         done_marker=f"{TL}/cub/cub_stage1_resnet18/pseudo_hierarchy.pt"),
    dict(tag="tambahan", desc="HiProbCBM Tahap 2 @ CUB @ ResNet18 (dual-backbone)",
         cmd=f"python scripts/run_stage2.py --config configs/cub_stage2_resnet18.yaml "
             f"--stage1-log-dir {TL}/cub/cub_stage1_resnet18",
         done_marker=f"{TL}/cub/cub_stage2_resnet18/stage2_last.pth"),
    dict(tag="tambahan", desc="HiProbCBM-A3 Tahap 1 (SAE BatchTopK) @ CUB",
         cmd="python scripts/run_stage1.py --config configs/cub_stage1_batchtopk_sae.yaml",
         done_marker=f"{TL}/cub/cub_stage1_batchtopk/pseudo_hierarchy.pt"),
    dict(tag="tambahan", desc="HiProbCBM-A3 Tahap 2 (SAE BatchTopK) @ CUB",
         cmd=f"python scripts/run_stage2.py --config configs/cub_stage2_batchtopk_sae.yaml "
             f"--stage1-log-dir {TL}/cub/cub_stage1_batchtopk",
         done_marker=f"{TL}/cub/cub_stage2_batchtopk_a3/stage2_last.pth"),
]

print(f"Total skenario terdaftar : {len(EXPERIMENTS)}")
print(f"Akan dijalankan (RUN_TAGS={RUN_TAGS}): {sum(1 for e in EXPERIMENTS if e['tag'] in RUN_TAGS)}")

In [ ]:
import subprocess
import time

results = []
to_run = [e for e in EXPERIMENTS if e["tag"] in RUN_TAGS]

for i, exp in enumerate(to_run, 1):
    marker = pathlib.Path("/content/HiProbCBM") / exp["done_marker"]
    print(f"\n{'='*80}\n[{i}/{len(to_run)}] ({exp['tag']}) {exp['desc']}\n{'='*80}")

    if marker.exists():
        print(f"SKIP -- sudah selesai sebelumnya ({marker})")
        results.append({**exp, "status": "skip (sudah ada)"})
        continue

    print(f"CMD: {exp['cmd']}")
    t0 = time.time()
    proc = subprocess.run(exp["cmd"], shell=True, cwd="/content/HiProbCBM")
    elapsed = (time.time() - t0) / 60

    if proc.returncode == 0 and marker.exists():
        status = f"SUKSES ({elapsed:.1f} menit)"
    elif proc.returncode == 0:
        status = f"SELESAI tapi file penanda tidak ditemukan -- cek manual ({elapsed:.1f} menit)"
    else:
        status = f"GAGAL (exit code {proc.returncode}, {elapsed:.1f} menit)"

    print(f"\n>>> {status}")
    results.append({**exp, "status": status})

print("\n\n" + "=" * 80)
print("RINGKASAN SEMUA SKENARIO")
print("=" * 80)
for r in results:
    print(f"[{r['tag']:8s}] {r['desc']:55s} -> {r['status']}")

n_fail = sum(1 for r in results if "GAGAL" in r["status"])
print(f"\nTotal dijalankan: {len(results)} | Gagal: {n_fail}")
if n_fail:
    print("Jalankan ulang sel ini -- yang sudah SUKSES otomatis di-skip, cuma yang GAGAL yang diulang.")
else:
    print("Semua skenario yang dipilih (RUN_TAGS) sudah selesai.")

## 14. Kumpulkan hasil akhir jadi CSV (bahan Tabel 4.7 & 4.8)

`hiprobcbm/engine/evaluate.py` sudah punya fungsi evaluasi (dipakai buat isi Tabel 4.7/4.8), tapi **belum ada CLI/sel yang memanggilnya** — jadi sel ini melengkapi bagian terakhir yang hilang: loop semua skenario yang **sudah SUKSES** (dicek dari file penanda yang sama seperti sel #13), evaluasi checkpoint-nya di test set, lalu tulis satu CSV gabungan langsung ke `train_log/` (yang sejak sel #12 sudah tersambung ke Drive, jadi CSV ini otomatis persisten, tanpa langkah salin manual).

**Catatan jujur:** sel ini baru ditulis sekarang, belum pernah dites lewat run sungguhan (karena training-nya sendiri belum pernah selesai) — sama seperti keseluruhan pipeline ini. Coba dulu dengan 1 skenario yang sudah selesai sebelum percaya hasil untuk semuanya.

In [ ]:
import re, csv
from hiprobcbm.config import load_config, resolve_device
from hiprobcbm.engine.evaluate import evaluate_baseline_checkpoint, evaluate_hiprobcbm_checkpoint

device = resolve_device(0)
rows = []

for exp in EXPERIMENTS:
    marker = pathlib.Path("/content/HiProbCBM") / exp["done_marker"]
    if not marker.exists():
        continue  # belum selesai training -- lewati, bukan bagian tabel hasil dulu

    config_path = re.search(r"--config (\S+)", exp["cmd"]).group(1)
    cfg = load_config(config_path)

    try:
        if "run_baseline.py" in exp["cmd"]:
            metrics = evaluate_baseline_checkpoint(cfg, marker, device)
        else:
            stage1_dir = pathlib.Path("/content/HiProbCBM") / re.search(r"--stage1-log-dir (\S+)", exp["cmd"]).group(1)
            hierarchy = torch.load(stage1_dir / "pseudo_hierarchy.pt", map_location="cpu", weights_only=False)
            metrics = evaluate_hiprobcbm_checkpoint(cfg, marker, hierarchy["subconcepts_per_concept"], device)

        metrics = {"skenario": exp["desc"], "tag": exp["tag"], **metrics}
        rows.append(metrics)
        print(f"OK    {exp['desc']}")
    except Exception as e:
        print(f"GAGAL {exp['desc']}: {e}")

out_csv = pathlib.Path("/content/HiProbCBM/train_log/ringkasan_hasil_bab4.csv")
if rows:
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(dict.fromkeys(k for r in rows for k in r.keys()))
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"\nTersimpan: {out_csv} ({len(rows)} baris)")
    print("train_log/ sudah tersambung ke Drive sejak sel #12, jadi CSV ini otomatis ikut persisten.")
else:
    print("Belum ada skenario yang SUKSES untuk dievaluasi -- jalankan sel #13 dulu.")